# ViLT PathVQA — Fast Training Pipeline
GPU-optimized: AMP, DataLoader workers, pin_memory, tqdm progress, early stopping.

In [77]:
import os
import re
import torch
from collections import Counter
from datasets import load_dataset, DatasetDict
from transformers import ViltProcessor, ViltForQuestionAnswering, get_cosine_schedule_with_warmup
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## Config

In [78]:
CACHE_DIR           = "/media/abood/NewVolume/huggingface_cache"   # fixed: removed space
SAVE_PATH           = "./best_model"
TOP_K_ANSWERS       = 2
BATCH_SIZE          = 8
ACCUM_STEPS         = 4        # effective batch = 64
NUM_WORKERS         = 4        # parallel data loading (was 0 — bottleneck)
NUM_EPOCHS          = 10
LEARNING_RATE       = 2e-4
WEGHT_DECAY  = 0.05
WARMUP_RATIO        = 0.1
GRAD_CLIP_NORM      = 1.0
EARLY_STOP_PATIENCE = 3
UNFREEZE_LAST_N     = 6

## 1. Load & Preprocess Data

In [79]:
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    return re.sub(r"\s+", " ", text)

print("Loading dataset...")
ds = load_dataset("flaviagiammarino/path-vqa", cache_dir="/media/abood/SAMSUNG/huggingface_cache")
ds = ds.map(lambda x: {"question": clean_text(x["question"]), "answer": clean_text(x["answer"])},
            num_proc=1)

# FIX: Explicitly force vocabulary to only contain yes and no
most_common   = ["yes", "no"]
answer2id     = {a: i for i, a in enumerate(most_common)}
id2answer     = {i: a for a, i in answer2id.items()}

# Filter out all non-yes/no answers + add binary labels (0 or 1)
valid = set(answer2id)
ds = DatasetDict({
    split: ds[split].filter(lambda x: x["answer"] in valid, num_proc=1)
                    .map(lambda x: {"label": answer2id[x["answer"]]}, num_proc=1)
    for split in ds
})
print(f"Answer classes: {len(answer2id)}")
for split in ds:
    print(f"  {split}: {len(ds[split]):,} samples")

Loading dataset...
Answer classes: 2
  train: 9,751 samples
  validation: 3,125 samples
  test: 3,362 samples


## 2. Tokenize with ViltProcessor

In [ ]:
from torchvision.transforms import ColorJitter, Compose, RandomHorizontalFlip, RandomVerticalFlip

processor = ViltProcessor.from_pretrained("dandelin/vilt-b32-mlm")

# Pathology images benefit heavily from flips and minor color variations
train_transforms = Compose([
    RandomHorizontalFlip(p=0.5),
    RandomVerticalFlip(p=0.5),
    #RandomRotation(degree = 15),
    ColorJitter(brightness=0.1, contrast=0.1),
])

def preprocess(batch, is_train=True):
    processed_imgs = []
    for img in batch["image"]:
        img = img.convert("RGB").resize((384, 384)) # Match ViLT standard resolution
        if is_train:
            img = train_transforms(img)
        processed_imgs.append(img)
        
    enc = processor(images=processed_imgs, text=batch["question"],
                     padding="max_length", truncation=True, max_length=40, return_tensors="pt")
    
    num_labels = TOP_K_ANSWERS
    labels = []
    for label in batch["label"]:
        one_hot = [0.0] * num_labels
        one_hot[label] = 1.0
        labels.append(one_hot)
    return {**enc, "labels": labels}

# Apply splits uniquely if mapping dynamically, or apply without train aug to val/test:
PROCESSED_CACHE = os.path.join("/media/abood/SAMSUNG/huggingface_cache", "processed_pathvqa_binary_v2")
cols = ["input_ids", "attention_mask", "token_type_ids", "pixel_values", "pixel_mask", "labels"]

if os.path.exists(PROCESSED_CACHE):
    print("Loading preprocessed data from disk cache...")
    from datasets import load_from_disk
    processed = load_from_disk(PROCESSED_CACHE)
else:
    print("Tokenizing binary dataset with high-res images...")
    # Map splits individually to ensure validation/testing doesn't use training augmentations
    processed = DatasetDict({
        "train": ds["train"].map(lambda b: preprocess(b, is_train=True), batched=True, batch_size=16, remove_columns=ds["train"].column_names),
        "validation": ds["validation"].map(lambda b: preprocess(b, is_train=False), batched=True, batch_size=16, remove_columns=ds["validation"].column_names),
        "test": ds["test"].map(lambda b: preprocess(b, is_train=False), batched=True, batch_size=16, remove_columns=ds["test"].column_names)
    })
    processed.save_to_disk(PROCESSED_CACHE)

processed.set_format(type="torch", columns=cols)

Tokenizing binary dataset with high-res images...


/home/abood/cuda_venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Saving the dataset (20/20 shards): 100%|██████████| 3362/3362 [02:21<00:00, 23.75 examples/s]


## 3. DataLoaders

In [82]:
def make_loader(split, shuffle):
    bs = BATCH_SIZE if shuffle else BATCH_SIZE * 2
    return DataLoader(
        processed[split],
        batch_size=bs,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,     # parallel workers (was 0 — big bottleneck)
        pin_memory=(device.type == "cuda"),   # faster CPU→GPU transfer
        persistent_workers=(NUM_WORKERS > 0), # keep workers alive between epochs
        prefetch_factor=2 if NUM_WORKERS > 0 else None,
    )

train_loader = make_loader("train",      shuffle=True)
val_loader   = make_loader("validation", shuffle=False)
test_loader  = make_loader("test",       shuffle=False)

print(f"Train batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")
print(f"Test batches  : {len(test_loader):,}")

Train batches : 1,219
Val batches   : 196
Test batches  : 211


## 4. Model Setup

In [83]:
model = ViltForQuestionAnswering.from_pretrained(
    "dandelin/vilt-b32-mlm",
    num_labels=TOP_K_ANSWERS,
    ignore_mismatched_sizes=True,
    use_safetensors=True,
    hidden_dropout_prob=0.2,           # Increased from 0.1
    attention_probs_dropout_prob=0.2,  # Increased from 0.1
)

# Freeze Embeddings
for p in model.vilt.embeddings.parameters():
    p.requires_grad = False

# Freeze initial layers
freeze_until = len(model.vilt.encoder.layer) - UNFREEZE_LAST_N
for i, block in enumerate(model.vilt.encoder.layer):
    if i < freeze_until:
        for p in block.parameters():
            p.requires_grad = False

model.to(device)
#model = ViltForQuestionAnswering.from_pretrained(
#    "dandelin/vilt-b32-mlm",
#    num_labels=TOP_K_ANSWERS,
#    ignore_mismatched_sizes=True,
#    use_safetensors=True,
#    hidden_dropout_prob=0.2,           # ADD
#    attention_probs_dropout_prob=0.2,  # ADD
#)

#for p in model.vilt.embeddings.parameters():
#    p.requires_grad = False

#freeze_until = len(model.vilt.encoder.layer) - UNFREEZE_LAST_N
#for i, block in enumerate(model.vilt.encoder.layer):
#    if i < freeze_until:
#        for p in block.parameters():
#            p.requires_grad = False

#model.to(device)

#trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
#total     = sum(p.numel() for p in model.parameters())
#print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
#print(f"Frozen blocks: 0-{freeze_until-1} | Trainable blocks: {freeze_until}-{len(model.vilt.encoder.layer)-1}")

Loading weights: 100%|██████████| 206/206 [00:00<00:00, 2527.40it/s]
[transformers] ViltForQuestionAnswering LOAD REPORT from: dandelin/vilt-b32-mlm
Key                                  | Status     | 
-------------------------------------+------------+-
mlm_score.transform.dense.bias       | UNEXPECTED | 
mlm_score.decoder.weight             | UNEXPECTED | 
mlm_score.bias                       | UNEXPECTED | 
mlm_score.transform.LayerNorm.bias   | UNEXPECTED | 
mlm_score.transform.dense.weight     | UNEXPECTED | 
mlm_score.transform.LayerNorm.weight | UNEXPECTED | 
classifier.{0, 1, 3}.bias            | MISSING    | 
classifier.{0, 1, 3}.weight          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ViltForQuestionAnswering(
  (vilt): ViltModel(
    (embeddings): ViltEmbeddings(
      (text_embeddings): TextEmbeddings(
        (word_embeddings): Embedding(30522, 768)
        (position_embeddings): Embedding(40, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (patch_embeddings): ViltPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32))
      )
      (token_type_embeddings): Embedding(2, 768)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (encoder): ViltEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViltLayer(
          (attention): ViltAttention(
            (attention): ViltSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_

## 5. Optimizer & Scheduler

In [84]:
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)

total_steps  = (len(train_loader) // ACCUM_STEPS) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler       = torch.amp.GradScaler("cuda")

print(f"Total steps  : {total_steps:,}")
print(f"Warmup steps : {warmup_steps:,}")

Total steps  : 3,040
Warmup steps : 304


## 6. Training Loop

In [85]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total, step = 0, 0, 0, 0

    pbar = tqdm(loader, desc="Train" if train else "Eval", leave=False)
    with torch.set_grad_enabled(train):
        if train:
            optimizer.zero_grad(set_to_none=True)  # faster than zero_grad()

        for batch in pbar:
            batch  = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

            if batch["labels"].ndim == 1:
                targets = batch["labels"].long()
                batch["labels"] = torch.nn.functional.one_hot(targets, num_classes=TOP_K_ANSWERS).float()
            else:
                targets = batch["labels"].argmax(-1)
                batch["labels"] = batch["labels"].float()

            with torch.amp.autocast("cuda"):
                out  = model(**batch)
                loss = out.loss / ACCUM_STEPS

            if train:
                scaler.scale(loss).backward()
                step += 1
                if step % ACCUM_STEPS == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        [p for p in model.parameters() if p.requires_grad], GRAD_CLIP_NORM
                    )
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
            
            total_loss += out.loss.item()
            preds       = out.logits.argmax(-1)
            correct    += (preds == targets).sum().item()
            total      += len(targets)
            pbar.set_postfix(loss=f"{out.loss.item():.3f}", acc=f"{correct/total:.3f}")

    return total_loss / len(loader), correct / total

In [86]:
best_val_loss, patience_count = float("inf"), 0

print("── Training ──")
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss,   val_acc   = run_epoch(val_loader,   train=False)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        patience_count = 0
        model.save_pretrained(SAVE_PATH)
        processor.save_pretrained(SAVE_PATH)
        torch.save({"answer2id": answer2id, "id2answer": id2answer},
                   os.path.join(SAVE_PATH, "vocab.pt"))
        print(f"  Saved best model (val_loss={val_loss:.4f})")
    else:
        patience_count += 1
        if patience_count >= EARLY_STOP_PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

print(f"\nBest Val Loss: {best_val_loss:.4f}")

── Training ──


Train:   0%|          | 0/1219 [00:00<?, ?it/s]

Epoch 1/10 | Train Loss: 1.0510 Acc: 0.6994 | Val Loss: 0.8462 Acc: 0.7798


Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.47s/it]


  Saved best model (val_loss=0.8462)


...

Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.04s/it]


  Saved best model (val_loss=0.6604)


Epoch 6/10 | Train Loss: 0.4990 Acc: 0.8841 | Val Loss: 0.6780 Acc: 0.8544


Epoch 7/10 | Train Loss: 0.4204 Acc: 0.9045 | Val Loss: 0.7864 Acc: 0.8608


Epoch 8/10 | Train Loss: 0.3597 Acc: 0.9213 | Val Loss: 0.8245 Acc: 0.8643
  Early stopping at epoch 8

Best Val Loss: 0.6604


## 7. Testing

In [87]:
print("── Testing ──")
model = ViltForQuestionAnswering.from_pretrained(SAVE_PATH).to(device)

test_loss, test_acc = run_epoch(test_loader, train=False)
print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")

── Testing ──


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 2374.94it/s]
                                                                              

Test Loss: 0.6413 | Test Accuracy: 0.8543


## 8. Results


In [88]:
#Experiments & Results

import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

#Collect predictions on test set
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Collecting predictions"):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        if batch["labels"].ndim == 1:
            targets = batch["labels"].long()
        else:
            targets = batch["labels"].argmax(-1)

        logits = model(**{k: v for k, v in batch.items() if k != "labels"}).logits
        preds  = logits.argmax(-1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

all_preds   = np.array(all_preds)
all_targets = np.array(all_targets)

#metrics summary table
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

acc  = accuracy_score(all_targets, all_preds)
f1   = f1_score(all_targets, all_preds, average="weighted", zero_division=0)
prec = precision_score(all_targets, all_preds, average="weighted", zero_division=0)
rec  = recall_score(all_targets, all_preds, average="weighted", zero_division=0)

print("=" * 45)
print("         Experiment Results Summary")
print("=" * 45)
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("=" * 45)

#Top-10 class report
label_names = [id2answer[i] for i in range(len(id2answer))]
print("\nPer-class Report (top answers):")
print(classification_report(all_targets, all_preds, target_names=label_names, 
                             zero_division=0, labels=list(range(len(label_names)))))

         Experiment Results Summary
  Accuracy  : 0.8543
  Precision : 0.8576
  Recall    : 0.8543
  F1-Score  : 0.8545

Per-class Report (top answers):
              precision    recall  f1-score   support

         yes       0.90      0.83      0.86      1816
          no       0.81      0.89      0.85      1546

    accuracy                           0.85      3362
   macro avg       0.85      0.86      0.85      3362
weighted avg       0.86      0.85      0.85      3362



## 8. Inference Example

In [89]:
def predict(image, question: str) -> str:
    """Single-sample prediction. Pass a PIL image and a question string."""
    vocab = torch.load(os.path.join(SAVE_PATH, "vocab.pt"))
    enc   = processor(
        images=image.convert("RGB").resize((128, 128)),
        text=clean_text(question),
        return_tensors="pt"
    ).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits
    return vocab["id2answer"][logits.argmax(-1).item()]

# Example usage:
# from PIL import Image
# img = Image.open("sample.jpg")
# answer = predict(img, "Is there a tumor present?")
# print("Answer:", answer)

In [90]:
from PIL import Image
img = Image.open("image.jpg")
answer = predict(img, "Is there a tumor present?")
print("Answer:", answer)

Answer: yes
